# Multi-head attention from scratch
**LLMs From Basics · M1.2 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/m1-2-multihead.ipynb)

In the lesson you saw “bank” split its attention across several heads. Here you'll build multi-head attention in NumPy and check every claim the lesson made:

1. **split** a $(n, d_{\text{model}})$ matrix into $h$ heads of size $d_k = d_{\text{model}}/h$, and **merge** them back
2. **multi-head attention**: per-head attention, concatenate, project with $W_O$
3. the **parameter count** doesn't depend on $h$ (and how GQA changes it)
4. a check against PyTorch's `nn.MultiheadAttention`
5. **grouped-query attention** (GQA), where several query heads share one key/value head

**How to use it:** run cells top to bottom (Shift+Enter). Each `# TODO` cell is followed by a ✅ check cell. Stuck for more than five minutes? The **Solutions** are at the bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

# From M1.1 (given): softmax, causal mask and single-head attention.
def softmax(x, axis=-1):
    z = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def causal_mask(n):
    return np.where(np.triu(np.ones((n, n), dtype=bool), k=1), -np.inf, 0.0)

def attention(Q, K, V, mask=None):
    # Works on (n, d) arrays or batched (..., n, d) arrays, e.g. (h, n, d_k).
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    w = softmax(scores, axis=-1)
    return w @ V, w

tokens = ["I", "sat", "by", "the", "river", "bank"]
n, d_model, h = len(tokens), 16, 4
X = rng.normal(size=(n, d_model))                      # stand-in token vectors
W_Q, W_K, W_V, W_O = (rng.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(4))
print("X:", X.shape, " each W:", W_Q.shape)

## 1. Split into heads, and merge back
`split_heads` takes $(n, d_{\text{model}})$ and returns $(h, n, d_k)$: head $i$ gets columns $i\,d_k$ to $(i+1)\,d_k - 1$.
The trick is a **reshape** to $(n, h, d_k)$ followed by a **transpose** that moves the head axis to the front. `merge_heads` undoes it.

In [ ]:
def split_heads(A, h):
    # (n, d_model) -> (h, n, d_k)
    # TODO: reshape A to (n, h, d_k), then transpose the first two axes.
    raise NotImplementedError("Fill in split_heads")

def merge_heads(A):
    # (h, n, d_k) -> (n, h * d_k), i.e. concatenate the heads side by side
    # TODO: transpose back to (n, h, d_k), then reshape to (n, h * d_k).
    raise NotImplementedError("Fill in merge_heads")

In [ ]:
Qh = split_heads(X @ W_Q, h)
assert Qh.shape == (h, n, d_model // h), Qh.shape
d_k = d_model // h
assert np.allclose(Qh[2], (X @ W_Q)[:, 2 * d_k:3 * d_k]), "Head 2 should be columns 2*d_k ... 3*d_k - 1."
assert np.allclose(merge_heads(Qh), X @ W_Q), "merge_heads(split_heads(A)) should give A back."
print("✅ split/merge work. Head shape:", Qh.shape)

## 2. Multi-head attention
$$\mathrm{head}_i = \mathrm{Attention}(XW_Q^{(i)}, XW_K^{(i)}, XW_V^{(i)}), \qquad \mathrm{MultiHead}(X) = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)\,W_O$$
Compute $Q = XW_Q$ (and $K$, $V$) with the full matrices, split them into heads, run `attention` **once** on the $(h, n, d_k)$ arrays (it broadcasts over the head axis), merge, and multiply by $W_O$.
Note: inside each head the scaling is $\sqrt{d_k}$, the *head* size. `attention` handles that automatically because it reads `Q.shape[-1]`.

In [ ]:
def multi_head_attention(X, W_Q, W_K, W_V, W_O, h, causal=True):
    # Returns (output (n, d_model), weights (h, n, n))
    n = X.shape[0]
    mask = causal_mask(n) if causal else None
    # TODO 1: Qh, Kh, Vh = split_heads(X @ W_Q, h), ... for K and V
    # TODO 2: out_h, w = attention(Qh, Kh, Vh, mask)        # out_h: (h, n, d_k)
    # TODO 3: return merge_heads(out_h) @ W_O, w
    raise NotImplementedError("Fill in multi_head_attention")

In [ ]:
out, w = multi_head_attention(X, W_Q, W_K, W_V, W_O, h)
assert out.shape == (n, d_model) and w.shape == (h, n, n), (out.shape, w.shape)
assert np.allclose(w.sum(-1), 1), "Every head's rows must sum to 1."
assert np.allclose(np.triu(w, k=1), 0), "The causal mask must hold in every head."

# With h = 1 it must be exactly single-head attention followed by W_O.
o1, _ = multi_head_attention(X, W_Q, W_K, W_V, W_O, 1)
ref, _ = attention(X @ W_Q, X @ W_K, X @ W_V, causal_mask(n))
assert np.allclose(o1, ref @ W_O)

# Head i really uses only its own slice of W_Q, W_K, W_V.
d_k = d_model // h
s = slice(1 * d_k, 2 * d_k)
_, w_head1 = attention(X @ W_Q[:, s], X @ W_K[:, s], X @ W_V[:, s], causal_mask(n))
assert np.allclose(w[1], w_head1), "Head 1's weights should come from columns d_k ... 2*d_k - 1."
print("✅ multi-head attention works. Output", out.shape, "weights", w.shape)

## 3. Counting parameters
Write `attn_param_count(d_model, h, n_kv=None)`: the number of weights in $W_Q, W_K, W_V, W_O$ (no biases).
With standard multi-head attention (`n_kv = None`, meaning $n_{kv} = h$), all four are $d_{\text{model}} \times d_{\text{model}}$.
With grouped-query attention, $W_K$ and $W_V$ only need $n_{kv} \cdot d_k$ output columns.

In [ ]:
def attn_param_count(d_model, h, n_kv=None):
    # TODO: d_k = d_model // h; n_kv defaults to h.
    #       W_Q and W_O: d_model * d_model each. W_K and W_V: d_model * (n_kv * d_k) each.
    raise NotImplementedError("Fill in attn_param_count")

In [ ]:
counts = {hh: attn_param_count(512, hh) for hh in [1, 2, 4, 8, 16]}
print("d_model = 512:", counts)
assert len(set(counts.values())) == 1 and counts[8] == 4 * 512**2 == 1_048_576, counts
assert attn_param_count(4096, 32) == 67_108_864                     # the lesson's faded example
assert attn_param_count(4096, 32, n_kv=8) == 2 * 4096**2 + 2 * 4096 * 1024   # GQA, as in Llama 3 8B
print("GQA (32 query heads, 8 KV heads):", f"{attn_param_count(4096, 32, n_kv=8):,}", "vs MHA", f"{attn_param_count(4096, 32):,}")
print("✅ the head count doesn't change the parameter count; GQA shrinks W_K and W_V")

## 4. Check against PyTorch (optional)
`torch.nn.MultiheadAttention` stores $W_Q, W_K, W_V$ stacked as one `in_proj_weight` of shape $(3d, d)$ and computes `x @ weight.T`, so we copy in the **transposes**. Your output should match to floating-point precision.

In [ ]:
try:
    import torch
    mha = torch.nn.MultiheadAttention(d_model, h, bias=False, batch_first=True).double()
    with torch.no_grad():
        mha.in_proj_weight.copy_(torch.from_numpy(np.concatenate([W_Q, W_K, W_V], axis=1).T))
        mha.out_proj.weight.copy_(torch.from_numpy(W_O.T))
    Xt = torch.from_numpy(X)[None]                                   # add a batch dimension
    future = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)  # True = not allowed
    ref, _ = mha(Xt, Xt, Xt, attn_mask=future, need_weights=False)
    mine, _ = multi_head_attention(X, W_Q, W_K, W_V, W_O, h)
    print("max |difference| vs PyTorch:", float(np.abs(ref[0].detach().numpy() - mine).max()))
    assert np.allclose(ref[0].detach().numpy(), mine, atol=1e-8)
    print("✅ matches torch.nn.MultiheadAttention")
except ImportError:
    print("PyTorch isn't installed here, so skipping this check. (It comes preinstalled on Colab.)")

## 5. Grouped-query attention
Now $W_K$ and $W_V$ have shape $(d_{\text{model}}, n_{kv} \cdot d_k)$, giving only $n_{kv}$ key/value heads. Query heads are split into $n_{kv}$ consecutive groups of $h / n_{kv}$; every head in a group uses the same K/V head.
The simplest implementation: split K and V into $n_{kv}$ heads, then **repeat** each one $h / n_{kv}$ times along the head axis (`np.repeat(..., axis=0)`), and run ordinary multi-head attention.

In [ ]:
def grouped_query_attention(X, W_Q, W_K, W_V, W_O, h, n_kv, causal=True):
    # W_K, W_V: (d_model, n_kv * d_k). Returns (output, weights (h, n, n)).
    n = X.shape[0]
    mask = causal_mask(n) if causal else None
    # TODO 1: Qh = split_heads(X @ W_Q, h); Kh = split_heads(X @ W_K, n_kv); same for Vh
    # TODO 2: repeat Kh and Vh h // n_kv times along axis 0 so they have h heads
    # TODO 3: attention, merge_heads, @ W_O (as before)
    raise NotImplementedError("Fill in grouped_query_attention")

In [ ]:
d_k = d_model // h
# n_kv = h must reproduce standard multi-head attention exactly.
o_full, _ = grouped_query_attention(X, W_Q, W_K, W_V, W_O, h, n_kv=h)
assert np.allclose(o_full, multi_head_attention(X, W_Q, W_K, W_V, W_O, h)[0])

# n_kv = 2: K/V projections are half as wide.
n_kv = 2
W_K2, W_V2 = W_K[:, :n_kv * d_k], W_V[:, :n_kv * d_k]
o2, w2 = grouped_query_attention(X, W_Q, W_K2, W_V2, W_O, h, n_kv)
assert o2.shape == (n, d_model) and w2.shape == (h, n, n)
# Heads 0 and 1 share K/V head 0, so with the same query they'd attend identically:
_, w_same = grouped_query_attention(X, np.tile(W_Q[:, :d_k], h), W_K2, W_V2, W_O, h, n_kv)
assert np.allclose(w_same[0], w_same[1]) and not np.allclose(w_same[1], w_same[2])

# MQA is just n_kv = 1.
o_mqa, _ = grouped_query_attention(X, W_Q, W_K[:, :d_k], W_V[:, :d_k], W_O, h, 1)
print("KV vectors cached per token per layer:  MHA", 2 * h * d_k, "| GQA(2)", 2 * n_kv * d_k, "| MQA", 2 * d_k)
print("✅ grouped-query attention works")

## Experiment: what do the heads look at?
Below, each head's causal attention pattern on our six tokens. The weights are **random** (untrained), so the patterns mean nothing yet; but notice the four heads already differ, because each has its own slice of $W_Q$ and $W_K$.
Try `h = 1`, `2`, `8`, and try scaling `W_Q` up by 5 (it makes every head's softmax sharper).

In [ ]:
h_exp = 4
_, w_exp = multi_head_attention(X, W_Q, W_K, W_V, W_O, h_exp)
fig, axes = plt.subplots(1, h_exp, figsize=(3.1 * h_exp, 3.2))
axes = np.atleast_1d(axes)
for i, ax in enumerate(axes):
    ax.imshow(w_exp[i], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"head {i + 1}")
    ax.set_xticks(range(n), tokens, rotation=90); ax.set_yticks(range(n), tokens if i == 0 else [])
plt.suptitle("Causal attention weights per head (random weights)")
plt.tight_layout(); plt.show()

**Think about it** (answers in the Solutions section):
1. Multi-head attention with $h$ heads computes $h$ score matrices of size $n \times n$. Why is the total multiply-add count for the scores the same as for one head with $d_k = d_{\text{model}}$?
2. Why can't you make $h$ as large as $d_{\text{model}}$ (so $d_k = 1$) and expect it to work well?
3. In GQA, which tensors shrink in the KV cache, and by what factor, going from 32 KV heads to 8?

---
## Solutions
Only run these if you're stuck. They redefine the functions; re-run the check cells afterwards.

In [ ]:
def split_heads(A, h):
    n, d_model = A.shape
    return A.reshape(n, h, d_model // h).transpose(1, 0, 2)

def merge_heads(A):
    h, n, d_k = A.shape
    return A.transpose(1, 0, 2).reshape(n, h * d_k)

In [ ]:
def multi_head_attention(X, W_Q, W_K, W_V, W_O, h, causal=True):
    n = X.shape[0]
    mask = causal_mask(n) if causal else None
    Qh, Kh, Vh = (split_heads(X @ W, h) for W in (W_Q, W_K, W_V))
    out_h, w = attention(Qh, Kh, Vh, mask)
    return merge_heads(out_h) @ W_O, w

In [ ]:
def attn_param_count(d_model, h, n_kv=None):
    d_k = d_model // h
    n_kv = h if n_kv is None else n_kv
    return 2 * d_model * d_model + 2 * d_model * (n_kv * d_k)

In [ ]:
def grouped_query_attention(X, W_Q, W_K, W_V, W_O, h, n_kv, causal=True):
    n = X.shape[0]
    mask = causal_mask(n) if causal else None
    Qh = split_heads(X @ W_Q, h)
    Kh = np.repeat(split_heads(X @ W_K, n_kv), h // n_kv, axis=0)
    Vh = np.repeat(split_heads(X @ W_V, n_kv), h // n_kv, axis=0)
    out_h, w = attention(Qh, Kh, Vh, mask)
    return merge_heads(out_h) @ W_O, w

**Think-about-it answers**
1. Each head does $n^2 \cdot d_k$ multiply-adds for $QK^\top$; $h$ heads do $n^2 \cdot h \cdot d_k = n^2 \cdot d_{\text{model}}$, the same as one head of width $d_{\text{model}}$. Splitting divides the work, it doesn't add to it.
2. With $d_k = 1$ each query and key is a single number, so a head can only express “bigger/smaller” matches and its softmax ranks keys by one scalar. Heads need enough dimensions to represent meaningful matches; in practice $d_k$ is 64 or 128.
3. The cached keys and values (one $k$ and one $v$ of size $d_k$ per KV head, per token, per layer) shrink by $32 / 8 = 4\times$. Queries are never cached.

**Next:** M1.3, positional encodings. Attention can't see word order on its own.